<a href="https://colab.research.google.com/github/VishnuCodes96/Rebounce-Applied-AI-and-Analytics-Codes/blob/main/python_301.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP3 Assignment

## Part 1 - Load and understand the data

importing numpy and pandas library

In [1]:
import numpy as np
import pandas as pd

Loading the dataset directly from the web, reason - dataset is not lost when the runtime restarts

In [2]:
url = "https://huggingface.co/datasets/aarav912/online-retail/resolve/main/online_retail.csv"
df = pd.read_csv(url)

The first 5 rows of dataset

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Number of rows and columns

In [4]:
print(f'No. of rows: {df.shape[0]}')
print(f'No. of columns: {df.shape[1]}')

No. of rows: 541909
No. of columns: 8


Names of the column

In [5]:
print(df.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')


Non-null items and Datatype of each column

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


Number of null items present in the dataset

In [7]:
df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


Number of Duplicate rows in the dataset

In [8]:
df.duplicated().sum()

np.int64(5268)

* **Customer ID & Description** columns have null values, so they need to be cleaned.

## Part 2 - Clean The Data

**Missing CustomerID's** - There are around 1.35 lakh missing customerId's, that's around 24% of the dataset, by dropping that many rows, the final analysis values could be distorted, so I would keep the rows that are missing the customerIds.

before cleaning, I would like to create a copy of the original, so as to avoid accidentally modifying the source data

In [9]:
clean_df = df.copy()

### Step 1
Checking whether the duplicates are genuine

In [10]:
clean_df[clean_df.duplicated(keep=False)].sort_values(by=['InvoiceNo', 'StockCode']).head(6)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom


`keep=false` is useful because it shows all instances of duplicated records rather than hiding the first occurrence.

from the above table, we can cearly see that there are duplicates (494 == 517, 485 == 539, 489 == 527), so can safely drop all duplicates.

In [11]:
print(f'No. of Rows before removing duplicates: {clean_df.shape[0]}')
clean_df = clean_df.drop_duplicates()
print(f'No. of Rows after removing duplicates: {clean_df.shape[0]}')

No. of Rows before removing duplicates: 541909
No. of Rows after removing duplicates: 536641


### Step 2
Checking Datatype of 'InvoiceDate' Column

In [12]:
print(clean_df['InvoiceDate'].dtype)

object


From the above cell, we can infer that 'InvoiceDate' column is an object datatype, we need to convert it into a proper datetime column, so as to perform calculations and analysis moving forward.

In [13]:
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
print(clean_df['InvoiceDate'].dtype)

datetime64[ns]


### Step 3
Checking if there are any **invalid transactions**.

In [14]:
invalid_rows = clean_df[(clean_df['Quantity'] <= 0) | (clean_df['UnitPrice'] <= 0)]
print(f'No. Of Invalid Transactions: {len(invalid_rows)}')

No. Of Invalid Transactions: 11763


Removing the Invalid Transaction Rows

In [15]:
clean_df = clean_df[(clean_df['Quantity'] > 0)]

the above cell, keeps only where both conditions are TRUE:
* `Quantity` is greater than 0
* `UnitPrice` is greater than 0

After removing Duplicates (5268 rows) and removing invalid transactions (11763 rows), the cleaned dataset should have `541909 - 5268 - 11763 = 524878` rows, the following cell checks that.

In [16]:
print(f'After cleaning the dataset, remaining number of valid rows: {clean_df.shape[0]}')

After cleaning the dataset, remaining number of valid rows: 526054


### Step 4
Creating a new sales column

In [17]:
clean_df['Sales'] = clean_df['Quantity'] * clean_df['UnitPrice']
clean_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Part 3 - Numpy Analysis

In [18]:
print(f'Mean of Sales in the dataset is: {round(np.mean(clean_df['Sales']),2)}')
print(f'Median of Sales in the dataset is: {round(np.median(clean_df['Sales']),2)}')
print(f'Minimum of Sales in the dataset is: {np.min(clean_df['Sales'])}')
print(f'Maximum of Sales in the dataset is: {round(np.max(clean_df['Sales']),2)}')
print(f'Standard Deviation of Sales in the dataset is: {round(np.std(clean_df['Sales']),2)}')

Mean of Sales in the dataset is: 20.19
Median of Sales in the dataset is: 9.92
Minimum of Sales in the dataset is: -11062.06
Maximum of Sales in the dataset is: 168469.6
Standard Deviation of Sales in the dataset is: 272.25


The mean (20.28) is more than twice the median (9.92). It suggests that the sales data is Right-Skewed.

The high standard deviation (271.69) compared to the mean shows that the sales values vary a lot, with some transactions having unusually high sales amounts.

The median gives a better idea of what a normal or typical transaction looks like than the mean.